# 🍄 Mushroom Classifier — YOLOv10 + DINOv2 Training (Colab)

This notebook trains **two new models** on the mushroom dataset:
- **YOLOv10n-cls** — Ultralytics NMS-free classification backbone
- **DINOv2-Small** — Facebook self-supervised ViT-S/14 backbone

Both follow the **exact same training methodology** as the existing models in the repo (same augmentations, loss functions, optimizer, early stopping).

### Before you start
1. `Runtime → Change runtime type → T4 GPU` ✅
2. Make sure you have a GitHub Personal Access Token with `repo` scope (for pushing the branch)
3. Have your Hugging Face token ready (for the mushroom dataset)

---

## 0. Verify GPU

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')

## 1. Clone the repo & checkout branch

In [ ]:
# ── Fill in your GitHub token ────────────────────────────────────────────────
GITHUB_TOKEN = 'ghp_YOUR_TOKEN_HERE'   # <-- replace this
GITHUB_USER  = 'QuasimodoCodes'        # <-- your GitHub username
GITHUB_EMAIL = 'you@example.com'       # <-- your email for commits
BRANCH       = 'feature/yolo10-dinov2'

import subprocess, os

!git clone https://{GITHUB_TOKEN}@github.com/QuasimodoCodes/Mushrooms.git
%cd Mushrooms
!git config user.name  "{GITHUB_USER}"
!git config user.email "{GITHUB_EMAIL}"
!git checkout {BRANCH}
!git log --oneline -5

## 2. Install dependencies

In [ ]:
# Core deps
!pip install -q ultralytics timm huggingface_hub python-dotenv tqdm

# DINOv2 via torch.hub doesn't need a separate install — it downloads on first use
# But we pre-warm the hub cache now to avoid timeout during training
import torch
print('Pre-loading DINOv2-Small from facebookresearch hub (~85 MB)...')
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=True)
print(f'DINOv2-Small loaded — embed dim: {backbone.embed_dim}')
del backbone

## 3. Download & prepare the dataset

In [ ]:
# ── Fill in your Hugging Face token ─────────────────────────────────────────
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # <-- replace this

import os
os.environ['HF_TOKEN'] = HF_TOKEN

# Run the existing dataset setup script (downloads + splits into train/val/test)
# This mirrors exactly what the other team members do locally.
!python scripts/setup/download_dataset.py

In [ ]:
# Quick sanity-check on the split
import os
data_dir = 'data/dataset_split'
for split in ['train', 'val', 'test']:
    split_path = os.path.join(data_dir, split)
    if os.path.isdir(split_path):
        n_classes = len(os.listdir(split_path))
        n_images  = sum(len(f) for _, _, f in os.walk(split_path))
        print(f'{split:>5}: {n_classes} classes, {n_images:,} images')
    else:
        print(f'WARNING: {split_path} not found!')

---
## 4. Train YOLOv10n-cls

Same structure as `scripts/training/yolo/train_yolo.py` but uses the v10 backbone.

In [ ]:
!python scripts/training/yolo10/train_yolo10.py

In [ ]:
# Show the training curve
import os, glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = 'docs/yolo10_runs/yolo10_classifier_v1'
pngs = glob.glob(os.path.join(run_dir, '*.png'))

fig, axes = plt.subplots(1, min(4, len(pngs)), figsize=(20, 5))
if len(pngs) == 1:
    axes = [axes]
for ax, p in zip(axes, pngs[:4]):
    ax.imshow(mpimg.imread(p))
    ax.set_title(os.path.basename(p))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
!python scripts/training/yolo10/evaluate_yolo10.py

---
## 5. Train DINOv2-Small

Same structure as `scripts/training/vit/train.py` — same epochs, augmentation, optimizer options, and CSV logging.

In [ ]:
# Default: AdamW + label-smoothed CE (same defaults as ViT run)
!python scripts/training/dinov2/train.py --optimizer adamw --loss ce_smooth

In [ ]:
# Plot training curves from results.csv
import pandas as pd
import matplotlib.pyplot as plt

csv_path = 'docs/franken_runs/dinov2_adamw_ce_smooth/results.csv'
df = pd.read_csv(csv_path)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DINOv2-Small Training', fontsize=14, fontweight='bold')

axes[0].plot(df['epoch'], df['train/loss'], label='train')
axes[0].plot(df['epoch'], df['val/loss'],   label='val')
axes[0].set_title('Loss');  axes[0].legend()

axes[1].plot(df['epoch'], df['train/top1'], label='train')
axes[1].plot(df['epoch'], df['metrics/accuracy_top1'], label='val')
axes[1].set_title('Top-1 Accuracy (%)');  axes[1].legend()

axes[2].plot(df['epoch'], df['train/top5'], label='train')
axes[2].plot(df['epoch'], df['metrics/accuracy_top5'], label='val')
axes[2].set_title('Top-5 Accuracy (%)');  axes[2].legend()

plt.tight_layout()
plt.savefig('docs/franken_runs/dinov2_adamw_ce_smooth/training_curves.png', dpi=150)
plt.show()
print(f"Best val top-1: {df['metrics/accuracy_top1'].max():.2f}%")

In [ ]:
# Evaluate on test set + generate confusion matrix
!python scripts/training/dinov2/evaluate.py --run dinov2_adamw_ce_smooth

---
## 6. Compare all models

In [ ]:
!python scripts/training/compare_all.py

---
## 7. Commit results & push to GitHub

This pushes the trained weights, results CSVs, and plots back to the `feature/yolo10-dinov2` branch.

In [ ]:
!git add docs/yolo10_runs/ docs/franken_runs/dinov2_adamw_ce_smooth/
!git add scripts/training/yolo10/ scripts/training/dinov2/
!git add scripts/training/compare_all.py
!git status

In [ ]:
!git commit -m "feat: add YOLOv10 and DINOv2 training results

- YOLOv10n-cls: trained 50 epochs on mushroom dataset_split
- DINOv2-Small (adamw + ce_smooth): trained 50 epochs with same
  hyperparameters as ViT-S/16 for direct comparison
- Both use identical augmentation, LR schedule, and early stopping
- Updated compare_all.py to include yolo10_runs and franken_runs

Trained on Google Colab T4 GPU"

!git push origin feature/yolo10-dinov2

---
## 8. (Optional) Download weights locally

If you want the model weights on your machine without cloning from GitHub:

In [ ]:
from google.colab import files

# Download YOLOv10 best weights
yolo10_best = 'docs/yolo10_runs/yolo10_classifier_v1/weights/best.pt'
if os.path.exists(yolo10_best):
    files.download(yolo10_best)

# Download DINOv2 best weights
dino_best = 'docs/franken_runs/dinov2_adamw_ce_smooth/weights/best.pt'
if os.path.exists(dino_best):
    files.download(dino_best)